# Library

In [1]:
import pandas as pd
import numpy as np
import pygad
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import os
from IPython.display import clear_output
import pygad
from classes import MACDBacktester, backtest, KalmanFilterEM


os.chdir(r'c:\\Users\\arvin\\Documents\\Coding Project\\V4\\Algotrading_RL\\src')
from utils.utils import CreateTimeFrames

# Data

In [ ]:
#os.getcwd()
os.chdir(r'c:\\Users\\arvin\\Documents\\Coding Project\\V4\\Algotrading_RL\\src\\data')
#os.getcwd()
df = pd.read_csv('BTCUSD_1m_2024-09-23.csv', index_col='time', usecols=['time', 'open', 'high', 'low', 'close', 'tick_volume'])
df.index = pd.to_datetime(df.index)

os.chdir(r'c:\\Users\\arvin\\Documents\\Coding Project\\V4\\Algotrading_RL\\src\\genetic')

price_column = 'close'
date_split = "2024-10-1"
tf = '1d'

timeframes = ['1min','5min','15min', '30min','1h', '4h','1d','1w','1m']
df = CreateTimeFrames(df,timeframes)

working_dataset = df[tf]

df_train = working_dataset[:date_split]
df_test = working_dataset[date_split:]

#signal_price = 'close'
real_price = 'close'

clear_output()


# Neuro genetic

In [3]:
class GAParameterOptimizer(nn.Module):
    def __init__(self, input_size, output_size):
        super(GAParameterOptimizer, self).__init__()
        self.fc1 = nn.Linear(input_size, 16)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(16, 32)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(32, output_size)
        self.sigmoid = nn.Sigmoid()  # To keep outputs in [0,1]

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu1(out)
        out = self.fc2(out)
        out = self.relu2(out)
        out = self.fc3(out)
        out = self.sigmoid(out)
        return out

class MACDOptimizerGA_new:
    def __init__(self, data_file, price_type='close'):
        """
        Initializes the MACDOptimizer class.

        Parameters:
        - data_file: Path to the CSV file containing the data or a pandas DataFrame.
        - price_type: The price column to use ('open', 'high', 'low', 'close').
        """
        self.data_file = data_file
        self.price_type = price_type
        self.data = self.load_data()

        # Initialize neural network for GA parameter optimization
        self.ga_param_nn = GAParameterOptimizer(input_size=1, output_size=5)
        self.criterion = nn.MSELoss()
        self.optimizer = optim.Adam(self.ga_param_nn.parameters(), lr=0.001)
        self.fitness_history = []
        self.ga_params_history = []

    def load_data(self):
        """
        Loads data from the input, which could be a CSV file path or a pandas DataFrame.
        """
        if isinstance(self.data_file, pd.DataFrame):
            full_data = self.data_file
        else:
            full_data = pd.read_csv(self.data_file)
            full_data.set_index('time', inplace=True)
        return full_data

    def calculate_macd(self, data, fast_period, slow_period, signal_period):
        """
        Calculates the MACD indicator.

        Parameters:
        - data: The DataFrame containing the price data.
        - fast_period: The period for the fast EMA.
        - slow_period: The period for the slow EMA.
        - signal_period: The period for the signal line EMA.
        """
        # Compute the EMAs
        ema_fast = data[self.price_type].ewm(span=fast_period, adjust=False).mean()
        ema_slow = data[self.price_type].ewm(span=slow_period, adjust=False).mean()
        # Calculate MACD components
        macd_line = ema_fast - ema_slow
        signal_line = macd_line.ewm(span=signal_period, adjust=False).mean()
        histogram = macd_line - signal_line
        # Add to DataFrame
        data['macd_line'] = macd_line
        data['signal_line'] = signal_line
        data['histogram'] = histogram
        return data

    def generate_signals(self, data):
        """
        Generates trading signals based on MACD crossover.

        Parameters:
        - data: The DataFrame containing the MACD data.
        """
        data['signal'] = 0
        data['signal'][1:] = np.where(
            data['macd_line'][1:] > data['signal_line'][1:], 1, 0
        )
        data['positions'] = data['signal'].diff()
        return data

    def backtest_strategy(self, data):
        """
        Backtests the trading strategy.

        Parameters:
        - data: The DataFrame containing the signals.
        """
        initial_capital = float(100000.0)
        positions = pd.DataFrame(index=data.index).fillna(0.0)
        positions['positions'] = data['signal'].shift(1).fillna(0)
        # Calculate holdings and cash
        positions['holdings'] = positions['positions'] * data[self.price_type]
        positions['cash'] = initial_capital - (
            (positions['positions'].diff() * data[self.price_type]).fillna(0).cumsum()
        )
        positions['total'] = positions['cash'] + positions['holdings']
        positions['returns'] = positions['total'].pct_change().fillna(0)
        total_return = positions['total'].iloc[-1] - initial_capital
        return total_return

    def optimize(self):
        """Runs the genetic algorithm to optimize MACD parameters and GA parameters."""

        def fitness_func(ga_instance, solution, solution_idx):
            """
            Fitness function for the genetic algorithm.

            Parameters:
            - ga_instance: The instance of the GA.
            - solution: The array of MACD parameters.
            - solution_idx: The index of the solution.
            """
            try:
                fast_period = int(solution[0])
                slow_period = int(solution[1])
                signal_period = int(solution[2])

                # Ensure valid periods
                if fast_period >= slow_period or fast_period <= 0 or slow_period <= 0 or signal_period <= 0:
                    return -np.inf  # Invalid solution

                # Copy data to avoid modifying the original
                data = self.data.copy()
                data = self.calculate_macd(data, fast_period, slow_period, signal_period)
                data = self.generate_signals(data)
                total_return = self.backtest_strategy(data)
                return total_return

            except Exception as e:
                print(f"Error in fitness function at solution index {solution_idx}: {e}")
                return -np.inf

        # Initial GA parameters
        for iteration in range(1):
            # Prepare input for neural network (e.g., previous best fitness)
            if self.fitness_history:
                nn_input = torch.tensor([[self.fitness_history[-1]]], dtype=torch.float32)
            else:
                nn_input = torch.tensor([[0.0]], dtype=torch.float32)

            # Neural network predicts GA parameters
            ga_params_normalized = self.ga_param_nn(nn_input).detach().numpy()[0]
            # Denormalize GA parameters to actual values
            num_generations = int(ga_params_normalized[0] * 100) + 10  # Range: 10 to 110
            sol_per_pop = int(ga_params_normalized[1] * 50) + 10        # Range: 10 to 60
            num_parents_mating = int(ga_params_normalized[2] * (sol_per_pop - 2)) + 2  # At least 2 parents
            mutation_probability = ga_params_normalized[3] * 0.5 + 0.01  # Range: 0.01 to 0.51
            crossover_probability = ga_params_normalized[4] * 0.5 + 0.5  # Range: 0.5 to 1.0

            # Ensure valid GA parameters
            num_generations = max(10, num_generations)
            sol_per_pop = max(10, sol_per_pop)
            num_parents_mating = min(num_parents_mating, sol_per_pop)
            mutation_probability = np.clip(mutation_probability, 0.01, 0.5)
            crossover_probability = np.clip(crossover_probability, 0.5, 1.0)

            # Record GA parameters
            ga_params = {
                'num_generations': num_generations,
                'sol_per_pop': sol_per_pop,
                'num_parents_mating': num_parents_mating,
                'mutation_probability': mutation_probability,
                'crossover_probability': crossover_probability
            }
            self.ga_params_history.append(ga_params)

            print(f"Iteration {iteration + 1}: GA Parameters:")
            print(ga_params)

            gene_space = [
                {'low': 2, 'high': 50, 'step': 1},   # fast_p eriod
                {'low': 5, 'high': 50, 'step': 1},   # slow_period
                {'low': 2, 'high': 30, 'step': 1},   # signal_period
            ]

            ga_instance = pygad.GA(
                num_generations=ga_params['num_generations'],
                num_parents_mating=ga_params['num_parents_mating'],
                fitness_func=fitness_func,
                sol_per_pop=ga_params['sol_per_pop'],
                num_genes=3,
                gene_space=gene_space,
                parent_selection_type="sss",
                keep_parents=2,
                crossover_type="single_point",
                mutation_type="random",
                mutation_probability=ga_params['mutation_probability'],
                crossover_probability=ga_params['crossover_probability'],
                mutation_num_genes=1,
            )

            ga_instance.run()

            solution, solution_fitness, solution_idx = ga_instance.best_solution()
            print("Best MACD Parameters:")
            print(f"Fast Period: {int(solution[0])}")
            print(f"Slow Period: {int(solution[1])}")
            print(f"Signal Period: {int(solution[2])}")
            print("Total Profit from Best Solution:", solution_fitness)

            # Record fitness
            self.fitness_history.append(solution_fitness)

            # Prepare training data for neural network
            if len(self.fitness_history) > 1:
                # Input: Previous fitness
                inputs = torch.tensor([[self.fitness_history[-2]]], dtype=torch.float32)
                # Target: Current GA parameters (normalized)
                targets = torch.tensor([[
                    (ga_params['num_generations'] - 10) / 100,
                    (ga_params['sol_per_pop'] - 10) / 50,
                    (ga_params['num_parents_mating'] - 2) / (ga_params['sol_per_pop'] - 2),
                    (ga_params['mutation_probability'] - 0.01) / 0.5,
                    (ga_params['crossover_probability'] - 0.5) / 0.5
                ]], dtype=torch.float32)

                # Zero the parameter gradients
                self.optimizer.zero_grad()
                # Forward pass
                outputs = self.ga_param_nn(inputs)
                # Compute loss
                loss = self.criterion(outputs, targets)
                # Backward pass and optimize
                loss.backward()
                self.optimizer.step()

        # Return the best solution from the last GA run
        return solution, solution_fitness


# EM KF

In [4]:
kf_em = KalmanFilterEM(df_train, price_column=real_price, max_iter=100, tol=1)

# Optimize Q and R using the EM algorithm
kf_em.optimize_em()

# Get smoothed data
smoothed_data = kf_em.get_smoothed_data()

df_train['smoothed_data'] = smoothed_data


optimizer = MACDOptimizerGA_new(data_file=df_train, price_type='smoothed_data')

best_solution, best_profit = optimizer.optimize()
#print(f"Optimization completed in {end_time - start_time} seconds.")
#clear_output()

Converged after 8 iterations.
Final Q: 1614621.029922721, Final R: 795281.1647349243


C:\Users\arvin\AppData\Local\Temp\ipykernel_12152\2799483076.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train['smoothed_data'] = smoothed_data


Iteration 1: GA Parameters:
{'num_generations': 57, 'sol_per_pop': 35, 'num_parents_mating': 20, 'mutation_probability': 0.2514975315332413, 'crossover_probability': 0.7712590396404266}


c:\Users\arvin\AppData\Local\Programs\Python\Python312\Lib\site-packages\pygad\pygad.py:1139: UserWarning: The 'delay_after_gen' parameter is deprecated starting from PyGAD 3.3.0. To delay or pause the evolution after each generation, assign a callback function/method to the 'on_generation' parameter to adds some time delay.
  warnings.warn("The 'delay_after_gen' parameter is deprecated starting from PyGAD 3.3.0. To delay or pause the evolution after each generation, assign a callback function/method to the 'on_generation' parameter to adds some time delay.")
C:\Users\arvin\AppData\Local\Temp\ipykernel_12152\4044199958.py:82: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are se

Best MACD Parameters:
Fast Period: 3
Slow Period: 14
Signal Period: 2
Total Profit from Best Solution: 6907.608517338638


C:\Users\arvin\AppData\Local\Temp\ipykernel_12152\4044199958.py:82: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  data['signal'][1:] = np.where(
C:\Users\arvin\AppData\Local\Temp\ipykernel_12152\4044199958.py:82: SettingWithCopyWarning: 
A v

# Test

In [5]:
print(f'Fast ema is:{best_solution[0]}, slow ema is:{best_solution[1]}, signal line is:{best_solution[2]}.')

kf_em_test = KalmanFilterEM(df_test, price_column=real_price, max_iter=100, tol=1)

# Optimize Q and R using the EM algorithm
kf_em_test.optimize_em()

# Get smoothed data
smoothed_data_test = kf_em_test.get_smoothed_data()

df_test['smoothed_data'] = smoothed_data_test
signal_price = 'smoothed_data'

backtester = MACDBacktester(df_test, best_solution[0], best_solution[1], best_solution[2], signal_price= signal_price)
backtester.calculate_macd()
backtester.generate_signals()
backtester.backtest_strategy()
metrics = backtester.get_performance_metrics()
print(metrics)
backtester.print_trades()

Fast ema is:3.0, slow ema is:14.0, signal line is:2.0.
Converged after 4 iterations.
Final Q: 934050.4248420605, Final R: 431906.86851324403
{'Total Return (%)': 10.441259346299617, 'Annualized Return (%)': 258.7614935667878, 'Annualized Volatility (%)': 23.74858129491496, 'Sharpe Ratio': 5.39287431927399, 'Max Drawdown (%)': 1.5826106406704814, 'Win Rate (%)': 100.0}
Trade 1: Return = 2.18%
Trade 2: Return = 8.09%
Total Return from trades: 10.27%


C:\Users\arvin\AppData\Local\Temp\ipykernel_12152\3991190757.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test['smoothed_data'] = smoothed_data_test
c:\Users\arvin\Documents\Coding Project\V4\Algotrading_RL\src\genetic\classes.py:99: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '99.88502791396199' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  self.data.at[idx, 'total'] = float(total)
c:\Users\arvin\Documents\Coding Project\V4\Algotrading_RL\src\genetic\classes.py:97: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '102.17735882623778